# qang desde cero: Las 15 compuertas y los 14 algoritmos en lenguaje qg (v0.14.0)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_inicio_formulacion.ipynb)

Cuaderno de inicio y fundamentos correspondiente a la obra reescrita *Computación, información y física cuántica en el espacio qang* (35 capítulos, Partes I–IV y fundamentos de la Parte VII).

### Principio Formal
Un estado de $n$ qubits se parametriza por sus **valores $qg$**, $qg_P = \langle P \rangle$ para cada cadena de Pauli $P$. Una compuerta unitaria $U$ opera mediante la transformación inducida:
$$qg'_P = qg_{U^\dagger P U}$$

### Contenido
0. Instalación y verificación del framework (`qang 0.14.0`).
1. El marco operativo: cálculo y reconstrucción $\rho = 2^{-n}\sum_P qg_P P$.
2. Las 15 compuertas cuánticas elementales: tablas analíticas y simetría de peso.
3. Los 14 algoritmos cuánticos principales como identidades de lectura $qg$.
4. Radio local de Bloch ($r_q^2$), entrelazamiento y corrección de deriva bajo $T_1$.

In [ ]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.14.0", "numpy", "matplotlib"], check=False)
importlib.invalidate_caches()

import numpy as np
import qang
from qang import formulation as F
from qang import sectors as S

print(f"Librería qang lista: v{qang.__version__}")

In [ ]:
# 1. Valores qg y reconstrucción de la matriz densidad
ket0 = np.array([1, 0])
theta = 0.7
qg_ry = F.qg_values(F.ry(theta) @ ket0)
print(f"|0>: {F.qg_values(ket0)}")
print(f"Ry(0.7)|0>: { {k: round(v, 4) for k, v in qg_ry.items()} } | cos(0.7) = {np.cos(theta):.4f} -> qg_Z = cos(theta)")
print(f"P(0) derivado de qg_Z: {F.born_p0(np.cos(theta)):.4f}")

# Estado de Bell
bell = F.apply_gate(F.apply_gate(F.qg_values(np.eye(4)[0]), np.kron(F.H, F.I2)), F.CX)
print(f"Estado de Bell (correlaciones puras): { {k: round(v, 3) for k, v in bell.items()} }")
print("Matriz densidad reconstruida:\n", np.round(F.state_from_qg(bell, 2).real, 3))

In [ ]:
# 2. Comprobación de compuertas y preservación de peso
def check_gate(name, U):
    tab = F.gate_table(U)
    pretty = {p: " ".join(f"{c:+.3g}·qg_{q}" for q, c in v.items()) for p, v in tab.items() if v != {p: 1.0}}
    n = int(np.log2(U.shape[0]))
    tag = "Clifford" if F.is_clifford(U) else "no Clifford"
    print(f"{name:12s} [{tag}, preserva peso: {F.conserves_weight(U, n)}]")

t = 0.7
gates = [("X", F.X), ("Z", F.Z), ("H", F.H), ("S", F.S), ("T", F.T), ("Ry(0.7)", F.ry(t)),
         ("CNOT", F.CX), ("CZ", F.CZ), ("SWAP", F.SWAP), ("iSWAP", F.ISWAP), ("Toffoli", F.TOFFOLI), ("Fredkin", F.FREDKIN)]
for name, U in gates:
    check_gate(name, U)

In [ ]:
# 3. Identidades de lectura de algoritmos
n = 4
const = F.phase_oracle_output(np.ones(2**n))
f_bal = np.array([1, -1] * 8)
print(f"1. Deutsch-Jozsa: Constante -> {F.deutsch_jozsa_is_constant(const, n)} | Balanceada -> {F.deutsch_jozsa_is_constant(F.phase_oracle_output(f_bal), n)}")

s = np.array([1, 0, 1, 1])
xs = (np.arange(2**n)[:, None] >> np.arange(n)[::-1]) & 1
p_bv = F.phase_oracle_output((-1.0) ** (xs @ s))
print(f"2. Bernstein-Vazirani: qg_Z = {[round(v, 2) for v in F.single_qg_z(p_bv, n)]} -> Secreto {F.bernstein_vazirani_secret(p_bv, n)}")

# Grover
n_g, marked = 5, 19
p = F.grover_state(n_g, marked, 4) ** 2
print(f"5. Grover (k=4 óptimo): Marcado = {F.grover_marked_from_signs(p, n_g)} | P(éxito) = {p[marked]:.4f}")

In [ ]:
# 4. Radio local de Bloch y entrelazamiento (Capítulos de Parte VII)
ket00 = np.array([1, 0, 0, 0])
bell_state = np.array([1, 0, 0, 1]) / np.sqrt(2)
print(f"|00> (Producto): r^2 = {F.radius_profile(ket00)} | Meyer-Wallach Q = {F.meyer_wallach(ket00):.2f}")
print(f"Bell (Máx. Entrelazado): r^2 = {F.radius_profile(bell_state)} | Meyer-Wallach Q = {F.meyer_wallach(bell_state):.2f}")